# 03 ANN・LIF-SNN・LUT の計算コスト比較

## 1. 研究目的

次の 3 つの計算の**コスト**（演算回数・パラメータ数・メモリ・実行時間）を、既存の Python 実装（CPU）で比べます。

1. **ANN** の forward 計算（$y = \mathrm{ReLU}(Wx+b)$）
2. **LIF-SNN** の時間発展計算（膜電位を $T$ ステップ更新する）
3. **Spiking Manifesto 型 LUT** の forward 計算（比較 → index → 行の読み出し → 加算）

> ### ⚠️ 比較の前に必ず読む
> この 3 つは**計算の内容が違います**。実行時間が同じでも、**同じ機能・同じ精度**という意味ではありません。
> 比較は次の 3 段階に分けて考えます。
>
> | 段階 | 内容 | この Notebook |
> | --- | --- | --- |
> | ① 計算原理の比較 | どんな演算を何回するか（乗算・加算・比較・参照）、何を保存し何を読むか | **中心** |
> | ② 実装の処理時間比較 | この Python 実装で何秒かかるか | **中心** |
> | ③ 同一タスクでの性能比較 | 同じ問題を同じ精度で解くのに必要なコスト | 扱わない（Notebook 05 Research E、README Advanced E） |

## 2. 理論的背景

| | ANN（入力 $n$、出力 $m$） | LIF 層（入力 $n$、ニューロン $m$、$T$ ステップ） | LUT（$n_t$ 個、各 $n_c$ 比較、出力 $m$） |
| --- | --- | --- | --- |
| 乗算 | $nm$ | $nm + 3mT$（※） | **0** |
| 加算 | $nm$ | $nm + 3mT$（※） | $n_t m$ |
| 比較 | $m$（ReLU） | $mT$（閾値） | $n_t n_c$ |
| LUT 参照 | 0 | 0 | $n_t$ |
| 保存するパラメータ | $nm+m$ | $nm+m$ | $n_t 2^{n_c} m$ |
| 1 回で読むパラメータ | $nm+m$（全部） | $nm+m$ | $n_t m$（選ばれた行だけ） |

※ この Notebook の **LIF 層**は「ANN と同じ $Wx+b$ を入力電流として $m$ 個の LIF ニューロンに与え、`lif_snn.simulate_lif` で $T$ ステップ時間発展させる」ものです。
1 ステップの更新 `v + dt/tau * (-(v - v_rest) + r*I)` を、ここでは**乗除算 3 回・加減算 3 回**と数えています（実装に依存する概算）。
LIF 層は**教育用に作った比較対象**で、論文のモデルではありません。

**計算量**：ANN は $O(nm)$、LIF 層は $O(nm + mT)$、LUT は $O(n_t(n_c+m))$。LUT の保存容量は $n_c$ に対して**指数関数的**（$2^{n_c}$）に増えます。

## 3. 実験環境（最初に必ず実行）

このセルは、E3 リポジトリの `.py` ファイル（`ann.py`、`spike_lut.py` など）を import できるように、
**作業ディレクトリ（E3 のルート）を自動で探して設定**します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのチームのリポジトリの URL** を入れる（`REPO_SUBDIR` に E3 のフォルダ名） |
| Colab（方法B：ZIP をアップロード） | E3 フォルダを ZIP にして、Colab の左の「ファイル」に `/content` へアップロードしておく（またはこのセルの実行中にアップロード） |

> ⚠️ `textbook/` フォルダ（教科書のコード）は**公開リポジトリには含めない**約束です。
> 方法A で `textbook/` がない場合、`spike_encoding.py`・`benchmark.py` の import が失敗します。
> その場合は方法B（ZIP）を使うか、授業で配布された `textbook/` を同じ場所に置いてください。

In [ ]:
import os
import sys
import glob
import zipfile
import subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                           # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "e3-ann-snn-comparison"   # リポジトリの中の E3 フォルダ（ルートが E3 なら "" にする）

IN_COLAB = "google.colab" in sys.modules


def is_e3_root(p):
    """spike_lut.py と ann.py があるフォルダを E3 のルートとみなす"""
    return (Path(p) / "spike_lut.py").exists() and (Path(p) / "ann.py").exists()


def find_e3_root():
    # 1) ローカル：notebooks/ の中、または E3 のルートで開いた場合
    for p in [Path.cwd(), Path.cwd().parent]:
        if is_e3_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    # 2) Colab 方法A：clone
    if REPO_URL:
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_e3_root(cand):
            return cand
    # 3) Colab 方法B：/content にある ZIP を展開（なければアップロードを求める）
    zips = glob.glob("/content/*.zip")
    if not zips:
        from google.colab import files
        print("E3 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/e3_upload")
    for hit in glob.glob("/content/e3_upload/**/spike_lut.py", recursive=True):
        if is_e3_root(Path(hit).parent):
            return Path(hit).parent
    return None


E3_ROOT = find_e3_root()
if E3_ROOT is None:
    raise RuntimeError("E3 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(E3_ROOT)
if str(E3_ROOT) not in sys.path:
    sys.path.insert(0, str(E3_ROOT))
print("E3_ROOT:", E3_ROOT)
if not (E3_ROOT / "textbook" / "insertion_sort.py").exists():
    print("⚠️ textbook/ がありません。spike_encoding.py などの import が失敗します（上の注意を参照）。")

In [ ]:
import matplotlib.pyplot as plt

# グラフの共通スタイル（Colab には日本語フォントがないことが多いので、図の文字は英語にしている）
plt.rcParams.update({
    "figure.figsize": (7, 3.5), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
})
# 色は役割ごとに固定する（ANN / LIF・SNN / LUT、CPU / GPU / TPU）
COLORS = {"ANN": "#2a78d6", "LIF": "#1baf7a", "LUT": "#eb6834",
          "CPU": "#2a78d6", "GPU": "#eb6834", "TPU": "#1baf7a", "neutral": "#8a8985"}

## 4. 使用するコード

| ファイル | 使う関数 |
| --- | --- |
| `ann.py` | `dense_forward`、`weighted_sum`、`ann_operation_counts` |
| `lif_snn.py` | `simulate_lif` |
| `spike_lut.py` | `lut_forward`、`make_random_tables`、`lut_operation_counts` |
| `benchmark.py` | `make_ann`（seed 固定の重み）、`traced_memory`（tracemalloc） |
| `tensor_backends.py` | `time_call`（ウォームアップ＋複数回測定の中央値） |

### 計測方法

- `time.perf_counter()` で 1 回ずつ測り、**ウォームアップ**（最初の数回は捨てる）のあと**複数回**測った**中央値**を使う
- 乱数シードを固定する（`SEED`）
- メモリは `tracemalloc`（Python オブジェクトとしての大きさ）と、**値の数 × 4 バイト（float32 を仮定）の理論値**を分けて記録する

In [ ]:
import platform
import random
import numpy as np

from ann import dense_forward, weighted_sum, ann_operation_counts
from lif_snn import simulate_lif
from spike_lut import lut_forward, make_random_tables, lut_operation_counts
from benchmark import make_ann, traced_memory
from tensor_backends import time_call

SEED = 0          # 乱数シード（結果と一緒に記録する）
WARMUP = 3        # ウォームアップ回数
REPEATS = 30      # 測定回数（中央値を使う）

print("Python", platform.python_version(), "/", platform.platform())

### LIF 層と、演算回数の関数

LIF 層は既存の `weighted_sum` と `simulate_lif` を組み合わせて作ります（新しいモデルは作りません）。

In [ ]:
def lif_layer_forward(x, W, b, T):
    """Wx + b を一定の入力電流として m 個の LIF ニューロンを T ステップ動かし、各ニューロンのスパイク数を返す"""
    currents = weighted_sum(x, W, b)
    return [simulate_lif([I] * T)["spike_count"] for I in currents]


def lif_operation_counts(n, m, T):
    """この Notebook の LIF 層の演算回数（1 ステップの更新を乗除算 3・加減算 3 と数えた概算）"""
    return {
        "multiplications": n * m + 3 * m * T,
        "additions": n * m + 3 * m * T,
        "comparisons": m * T,
        "lut_accesses": 0,
        "parameters": n * m + m,
        "values_read": n * m + m,
    }


def make_problem(n, m, nt, nc, seed=SEED):
    """同じ seed で、入力 x（0〜1 の乱数）・ANN の重み・LUT を作る"""
    rng = random.Random(seed)
    x = [rng.random() for _ in range(n)]
    W, b = make_ann(n, m, seed)
    tables = make_random_tables(n, nt, nc, m, seed=seed)
    return x, W, b, tables

## 5. 実行例：1 つの設定で 3 方式を比べる

$n = m = 32$、LIF は $T = 50$ ステップ、LUT は $n_t = 8$、$n_c = 6$。

In [ ]:
n, m, T, nt, nc = 32, 32, 50, 8, 6
x, W, b, tables = make_problem(n, m, nt, nc)

timings = {
    "ANN": time_call(lambda: dense_forward(x, W, b), warmup=WARMUP, repeats=REPEATS),
    "LIF": time_call(lambda: lif_layer_forward(x, W, b, T), warmup=WARMUP, repeats=REPEATS),
    "LUT": time_call(lambda: lut_forward(x, tables), warmup=WARMUP, repeats=REPEATS),
}
ops = {
    "ANN": ann_operation_counts(n, m),
    "LIF": lif_operation_counts(n, m, T),
    "LUT": lut_operation_counts(nt, nc, m),
}
mem_python = {
    "ANN": traced_memory(lambda: make_ann(n, m, SEED)),
    "LIF": traced_memory(lambda: make_ann(n, m, SEED)),   # LIF 層の保存パラメータは ANN と同じ W, b
    "LUT": traced_memory(lambda: make_random_tables(n, nt, nc, m, seed=SEED)),
}

print(f"n={n}, m={m}, T={T}, nt={nt}, nc={nc}, seed={SEED}, warmup={WARMUP}, repeats={REPEATS}")
keys = ["multiplications", "additions", "comparisons", "lut_accesses", "parameters", "values_read"]
print(f"{'':24}" + "".join(f"{k:>12}" for k in timings))
for key in keys:
    print(f"{key:24}" + "".join(f"{ops[k].get(key, 0):>12,}" for k in timings))
print(f"{'theory memory [bytes]':24}" + "".join(f"{4 * ops[k]['parameters']:>12,}" for k in timings))
print(f"{'Python memory [bytes]':24}" + "".join(f"{mem_python[k]:>12,}" for k in timings))
print(f"{'median time [us]':24}" + "".join(f"{timings[k]['median'] * 1e6:>12.1f}" for k in timings))

## 6. 結果の可視化

### 6-1. 実行時間の棒グラフと、演算回数の比較

In [ ]:
names = list(timings)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))

ax1.bar(names, [timings[k]["median"] * 1e6 for k in names], color=[COLORS[k] for k in names])
ax1.set_ylabel("median time [us]"); ax1.set_title("forward time (pure Python, CPU)")

op_keys = ["multiplications", "additions", "comparisons", "lut_accesses"]
width = 0.25
for i, k in enumerate(names):
    vals = [max(ops[k].get(key, 0), 0.8) for key in op_keys]   # 0 は log 軸に描けないので 0.8 として描く
    ax2.bar(np.arange(len(op_keys)) + (i - 1) * width, vals, width, color=COLORS[k], label=k)
ax2.set_yscale("log"); ax2.set_xticks(range(len(op_keys)))
ax2.set_xticklabels(["mult", "add", "compare", "LUT access"])
ax2.set_title("operation counts (bars at 0.8 mean zero)"); ax2.legend()
plt.tight_layout(); plt.show()

### 6-2. 入力次元 $n$ と実行時間

$m = n$、LUT は $n_t=8$、$n_c=6$ に固定。理論上、ANN は $n^2$、LUT は $n$ に比例して増えるはずです（LUT の比較回数は $n$ に依存しない）。

In [ ]:
dims = [8, 16, 32, 64, 128]
sweep_n = {"ANN": [], "LIF": [], "LUT": []}
for d in dims:
    x_, W_, b_, tables_ = make_problem(d, d, 8, 6)
    sweep_n["ANN"].append(time_call(lambda: dense_forward(x_, W_, b_), warmup=WARMUP, repeats=REPEATS)["median"])
    sweep_n["LIF"].append(time_call(lambda: lif_layer_forward(x_, W_, b_, T), warmup=1, repeats=10)["median"])
    sweep_n["LUT"].append(time_call(lambda: lut_forward(x_, tables_), warmup=WARMUP, repeats=REPEATS)["median"])

plt.figure()
for k, v in sweep_n.items():
    plt.loglog(dims, np.array(v) * 1e6, marker="o", ms=6, color=COLORS[k], label=k)
plt.xlabel("input dim n (= output dim m)"); plt.ylabel("median time [us]")
plt.title(f"time vs input dim (T={T}, nt=8, nc=6)"); plt.legend(); plt.show()

### 6-3. LUT 数 $n_t$ と実行時間、anchor pair 数 $n_c$ とメモリ使用量

In [ ]:
n_fixed = m_fixed = 32
nts = [1, 2, 4, 8, 16, 32]
t_nt = []
for k_ in nts:
    x_, _, _, tables_ = make_problem(n_fixed, m_fixed, k_, 6)
    t_nt.append(time_call(lambda: lut_forward(x_, tables_), warmup=WARMUP, repeats=REPEATS)["median"])

ncs = [1, 2, 4, 6, 8, 10]
mem_theory, mem_traced = [], []
for c in ncs:
    mem_theory.append(4 * lut_operation_counts(4, c, m_fixed)["parameters"])
    mem_traced.append(traced_memory(lambda: make_random_tables(n_fixed, 4, c, m_fixed, seed=SEED)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(nts, np.array(t_nt) * 1e6, marker="o", ms=6, color=COLORS["LUT"])
ax1.set_xlabel("number of LUTs nt"); ax1.set_ylabel("median time [us]")
ax1.set_title(f"LUT forward time vs nt (n=m={n_fixed}, nc=6)")
ax2.semilogy(ncs, mem_traced, marker="o", ms=6, color=COLORS["LUT"], label="tracemalloc (Python objects)")
ax2.semilogy(ncs, mem_theory, marker="s", ms=6, ls="--", color=COLORS["neutral"], label="theory: values x 4 bytes")
ax2.set_xlabel("anchor pairs per LUT nc"); ax2.set_ylabel("bytes (log)")
ax2.set_title(f"LUT memory vs nc (nt=4, m={m_fixed})"); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 6-4. LIF の時間ステップ数 $T$ と実行時間

In [ ]:
Ts = [10, 25, 50, 100, 200]
x_, W_, b_, _ = make_problem(32, 32, 1, 1)
t_T = [time_call(lambda: lif_layer_forward(x_, W_, b_, t), warmup=1, repeats=10)["median"] for t in Ts]
t_ann = time_call(lambda: dense_forward(x_, W_, b_), warmup=WARMUP, repeats=REPEATS)["median"]

plt.figure()
plt.plot(Ts, np.array(t_T) * 1e6, marker="o", ms=6, color=COLORS["LIF"], label="LIF layer")
plt.axhline(t_ann * 1e6, color=COLORS["ANN"], ls="--", label="ANN (no time steps)")
plt.xlabel("simulation steps T"); plt.ylabel("median time [us]")
plt.title("LIF layer time grows with T (n=m=32)"); plt.legend(); plt.show()

## 7. 結果の解釈（読み方のヒント）

- **演算回数（①計算原理）**：LUT は乗算 0 回です。LIF 層は時間ステップ $T$ に比例して計算が増えます。ANN は $nm$ に比例します。
- **保存容量と読み出し量**：LUT は保存するパラメータが多くなりがち（$2^{n_c}$ に比例）ですが、1 回の forward で読むのは選ばれた行だけ（$n_t m$）です。
- **実行時間（②実装）**：ここで測ったのは **Python の for ループとリスト**で書いた実装の時間です。NumPy や GPU を使うと大小関係が変わることがあります（Notebook 04）。
- **メモリ**：`tracemalloc` の値は Python の float・list の管理領域を含むので、理論値（値の数 × 4 バイト）より何倍も大きくなります。
- **③同一タスクでの性能**はこの Notebook では比べていません。ここでの LUT の行は学習していない乱数で、ANN の重みも乱数です。

> この測定から「SNN の方が速い・省電力」とは結論できません。消費電力は測っていません。

## 8. 学生向け TODO

- [ ] **TODO 1**：入力次元 `n` を変更する（6-2 の `dims`）。
- [ ] **TODO 2**：出力次元 `m` を $n$ と別に変更する（`make_problem(n, m, ...)`）。
- [ ] **TODO 3**：LIF の時間ステップ数 `T` を変更する（6-4 の `Ts`）。
- [ ] **TODO 4**：LUT 数 `nt` を変更する（6-3 の `nts`）。
- [ ] **TODO 5**：anchor pair 数 `nc` を変更する（6-3 の `ncs`。$n_c$ を大きくしすぎるとメモリが足りなくなるので 14 程度までにする）。
- [ ] **TODO 6**：理論計算量（演算回数の式）と実測時間を同じグラフに描き、比例関係を確かめる（例：時間 ÷ 演算回数）。
- [ ] **TODO 7**：`WARMUP`・`REPEATS` を変えて、中央値と四分位範囲（`np.percentile(times, [25, 75])`）を記録する。

In [ ]:
# TODO 6 の例：ANN の時間 / 乗算回数 を n ごとに計算してみよう
for d, t in zip(dims, sweep_n["ANN"]):
    ops_ = ann_operation_counts(d, d)["multiplications"]
    print(f"n={d:4d}: time/mult = {t / ops_ * 1e9:.2f} ns")   # TODO：LUT・LIF についても同様に

## 9. 考察課題

1. ANN と LUT で、入力次元 $n$ を増やしたときの時間の増え方はなぜ違いますか。演算回数の式から説明してください。
2. LUT の保存容量は $n_c$ とともに指数関数的に増えます。読み出し量（$n_t m$）が小さいことは、どんなハードウェアで有利になり得ますか。それはこの実験で確かめられていますか。
3. LIF 層の時間は $T$ に比例します。論文の LUT モデルが「時間ステップのシミュレーションをしない」ことには、計算コストの面でどんな意味がありますか。
4. 「実行時間が同じなら同じ性能」と言えないのはなぜですか。①〜③ の区別を使って説明してください。

## 10. 参考文献

- E. Izhikevich, "Spiking Manifesto", arXiv:2512.11843, 2025. <https://arxiv.org/abs/2512.11843>（arXiv の表示ライセンス：CC BY-NC-SA 4.0）
- 著者の公開コード：<https://github.com/izhikevich/SNN>（この教材には**コピーしていません**）
- W. Gerstner, W. M. Kistler, R. Naud, L. Paninski, *Neuronal Dynamics*, Cambridge University Press, 2014（LIF モデル）
- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 探索、第5章 再帰、第6章 ソート）
- E3 リポジトリの `README.md`（特に 17章「論文との対応関係」）

> この Notebook は論文の中心的なアイデアを**教育用に簡略化**したものです。論文のモデル・著者の実装そのものではありません。